In [43]:
import pandas as pd
import requests
from io import StringIO
import matplotlib.pyplot as plt

### ***Coletando os dados de cada jogo***

In [44]:
url_dota2 = "https://steamcharts.com/app/570"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(
    url_dota2,
    headers=headers
)

html = StringIO(response.text)

df_dota2 = pd.read_html(html)[0]

df_dota2.head()

,Month,Avg. Players,Gain,% Gain,Peak Players
0,Last 30 Days,425610.97,-307.6,-0.07%,695006
1,April 2026,425918.58,-51936.99,-10.87%,695006
2,March 2026,477855.57,-119154.25,-19.96%,859016
3,February 2026,597009.82,27685.64,+4.86%,869734
4,January 2026,569324.18,-24965.76,-4.20%,855690


In [45]:
url_smite = "https://steamcharts.com/app/386360"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(
    url_smite,
    headers=headers
)

html = StringIO(response.text)

df_smite = pd.read_html(html)[0]

df_smite.head()

,Month,Avg. Players,Gain,% Gain,Peak Players
0,Last 30 Days,1171.15,-27.0,-2.25%,2049
1,April 2026,1198.11,-13.42,-1.11%,2074
2,March 2026,1211.53,-80.66,-6.24%,2162
3,February 2026,1292.18,-149.29,-10.36%,2218
4,January 2026,1441.47,116.68,+8.81%,2607


In [46]:
url_lol = "https://activeplayer.io/league-of-legends/"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(
    url_lol,
    headers=headers
)

html = StringIO(response.text)

tabelas_lol = pd.read_html(html)

len(tabelas_lol)

for i, tabela in enumerate(tabelas_lol):
    print(f'Tabela {i}')
    print(tabela.head())
    print('\n')

Tabela 0
              0                                       1
0         Title                       League of Legends
1     Developer                              Riot Games
2     Publisher                              Riot Games
3  Release Date                        October 27, 2009
4        Genres  Multiplayer Online Battle Arena (MOBA)


Tabela 1
      Month   Players  Change
0  Apr 2026  16413143  -22.2%
1  Mar 2026  21100073  -11.3%
2  Feb 2026  23785537  -18.5%
3  Jan 2026  29202369  +17.6%
4  Dec 2025  24839037  -37.2%


Tabela 2
           0              1
0  Platform:  Availability:
1    Windows             ✔️
2      Linux              ❌
3        MAC             ✔️
4   XBOX ONE              ❌




In [47]:
df_lol = tabelas_lol[1]
df_lol.head()

,Month,Players,Change
0,Apr 2026,16413143,-22.2%
1,Mar 2026,21100073,-11.3%
2,Feb 2026,23785537,-18.5%
3,Jan 2026,29202369,+17.6%
4,Dec 2025,24839037,-37.2%


In [48]:
url_hots = "https://activeplayer.io/heroes-of-the-storm/"
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(
    url_hots,
    headers=headers
)

html = StringIO(response.text)

tabelas_hots = pd.read_html(html)

len(tabelas_hots)

for i, tabela in enumerate(tabelas_hots):
    print(f'Tabela {i}')
    print(tabela.head())
    print('\n')

Tabela 0
               0                       1
0    Game Title:     Heroes of the Storm
1     Developer:  Blizzard Entertainment
2     Publisher:  Blizzard Entertainment
3  Release Date:            June 2, 2015
4         Genre:  Moba, Action, Strategy


Tabela 1
           Month  Monthly Average Users  Gain / Loss  Gain / Loss %  \
0   Last 30 Days                    NaN     -1942663           -100   
1     March 2025              1942663.0      -632447            -25   
2  February 2025              2575110.0      -751844            -23   
3   January 2025              3326954.0       166481              5   
4  December 2024              3160473.0       -85566             -3   

   Daily Average Users  
0                    0  
1               556897  
2               738198  
3               953727  
4               906002  


Tabela 2
              0                    1
0     Game Name  Heroes of the Storm
1     Followers                 2.7M
2   Watch Hours               57527

In [49]:
df_hots = tabelas_hots[1]
df_hots.head()

,Month,Monthly Average Users,Gain / Loss,Gain / Loss %,Daily Average Users
0,Last 30 Days,NaN,-1942663,-100,0
1,March 2025,1942663.0,-632447,-25,556897
2,February 2025,2575110.0,-751844,-23,738198
3,January 2025,3326954.0,166481,5,953727
4,December 2024,3160473.0,-85566,-3,906002


### ***Normalização em Base 100***

In [ ]:
#Função para converter as datas

def converter_mes(coluna):

    data = pd.to_datetime(
        coluna,
        format='%B %Y',
        errors='coerce'
    )

    faltantes = data.isna()

    data.loc[faltantes] = pd.to_datetime(
        coluna[faltantes],
        format='%b %Y',
        errors='coerce'
    )

    return data

#Padronizar colunas

df_lol['Valor'] = df_lol['Players']

df_dota2['Valor'] = df_dota2['Avg. Players']

df_smite['Valor'] = df_smite['Avg. Players']

df_hots['Valor'] = df_hots['Monthly Average Users']

#Função para preparar dataframes

def preparar_dataframe(df, coluna_valor):

    df = df.copy()

    #Remover linha inválida
    df = df[df['Month'] != 'Last 30 Days']

    # Converter datas
    df['Month'] = converter_mes(df['Month'])

    #Converter números
    df[coluna_valor] = pd.to_numeric(
        df[coluna_valor],
        errors='coerce'
    )

    #Remover nulos
    df = df.dropna(subset=['Month', coluna_valor])

    #Ordenar datas
    df = df.sort_values('Month')

    #Garantir que não esteja vazio
    if df.empty:
        return df

    #Base 100
    valor_inicial = df[coluna_valor].iloc[0]

    df['Indice'] = (
        df[coluna_valor] / valor_inicial
    ) * 100

    return df

#Preparar dataframes

df_lol = preparar_dataframe(df_lol, 'Valor')

df_dota2 = preparar_dataframe(df_dota2, 'Valor')

df_smite = preparar_dataframe(df_smite, 'Valor')

df_hots = preparar_dataframe(df_hots, 'Valor')

#Resultados

print(df_lol[['Month', 'Valor', 'Indice']].head())

print(df_dota2[['Month', 'Valor', 'Indice']].head())

print(df_smite[['Month', 'Valor', 'Indice']].head())

print(df_hots[['Month', 'Valor', 'Indice']].head())

        Month     Valor      Indice
11 2025-05-01  13178039  100.000000
10 2025-06-01  20574951  156.130597
9  2025-07-01  23643626  179.416877
8  2025-08-01  29634339  224.876698
7  2025-09-01  35177744  266.942176
         Month      Valor      Indice
166 2012-07-01   52721.05  100.000000
165 2012-08-01   55768.61  105.780537
164 2012-09-01   61867.68  117.349104
163 2012-10-01   75965.44  144.089391
162 2012-11-01  101077.43  191.721201
         Month     Valor      Indice
128 2015-09-01   8041.89  100.000000
127 2015-10-01   9430.61  117.268577
126 2015-11-01   7849.60   97.608895
125 2015-12-01   7736.94   96.207981
124 2016-01-01  10017.13  124.561888
        Month      Valor      Indice
19 2023-09-01  2970690.0  100.000000
18 2023-10-01  3964841.0  133.465323
17 2023-11-01  4867813.0  163.861359
16 2023-12-01  3588358.0  120.792072
15 2024-01-01  3882486.0  130.693071


In [ ]:
#Cria a coluna com o nome do jogo
df_lol['Jogo'] = 'League of Legends'

df_dota2['Jogo'] = 'Dota 2'

df_smite['Jogo'] = 'SMITE'

df_hots['Jogo'] = 'Heroes of the Storm'

#Une os dataframes em um único dataframe
df_mobas = pd.concat(
    [
        df_lol,
        df_dota2,
        df_smite,
        df_hots
    ],
    ignore_index=True
)

df_mobas = df_mobas[
    ['Month', 'Valor', 'Indice', 'Jogo']
]

print('DataFrame:\n')
print(df_mobas.head(100))

print('\nDescrição:\n')
print(df_mobas.describe())

print('\nInformações:')
df_mobas.info()

print('\nValores Nulos:\n')
print(df_mobas.isnull().sum())

#Salvando o dataframe final em um arquivo CSV
df_mobas.to_csv(
    'df_mobas.csv',
    index=False
)

DataFrame:

        Month        Valor      Indice               Jogo
0  2025-05-01  13178039.00  100.000000  League of Legends
1  2025-06-01  20574951.00  156.130597  League of Legends
2  2025-07-01  23643626.00  179.416877  League of Legends
3  2025-08-01  29634339.00  224.876698  League of Legends
4  2025-09-01  35177744.00  266.942176  League of Legends
..        ...          ...         ...                ...
95 2019-06-01    507528.44  962.667549             Dota 2
96 2019-07-01    464787.61  881.597787             Dota 2
97 2019-08-01    467148.29  886.075467             Dota 2
98 2019-09-01    421971.24  800.384742             Dota 2
99 2019-10-01    388355.86  736.623910             Dota 2

[100 rows x 4 columns]

Descrição:

                            Month         Valor       Indice
count                         325  3.250000e+02   325.000000
mean   2020-07-12 09:40:25.846153  1.394181e+06   498.370510
min           2012-07-01 00:00:00  1.198110e+03    14.898363
25%        

### ***Tratamento de Dados***

In [79]:
#Removendo dados anteriores a maio de 2025
df_mobas = df_mobas[
    df_mobas['Month'] >= '2025-05-01'
]

print(df_mobas['Month'].min())

for jogo in df_mobas['Jogo'].unique():

    mask = df_mobas['Jogo'] == jogo

    valor_inicial = df_mobas.loc[
        mask,
        'Valor'
    ].iloc[0]

    df_mobas.loc[mask, 'Indice'] = (
        df_mobas.loc[mask, 'Valor'] / valor_inicial
    ) * 100

df_mobas.to_csv(
    'df_mobas_datas_corrigidas.csv',
    index=False
)

print(df_mobas.head(200))


2025-05-01 00:00:00
         Month        Valor      Indice               Jogo
0   2025-05-01  13178039.00  100.000000  League of Legends
1   2025-06-01  20574951.00  156.130597  League of Legends
2   2025-07-01  23643626.00  179.416877  League of Legends
3   2025-08-01  29634339.00  224.876698  League of Legends
4   2025-09-01  35177744.00  266.942176  League of Legends
5   2025-10-01  33170729.00  251.712178  League of Legends
6   2025-11-01  39531642.00  299.981219  League of Legends
7   2025-12-01  24839037.00  188.488113  League of Legends
8   2026-01-01  29202369.00  221.598745  League of Legends
9   2026-02-01  23785537.00  180.493752  League of Legends
10  2026-03-01  21100073.00  160.115424  League of Legends
11  2026-04-01  16413143.00  124.549206  League of Legends
166 2025-05-01    408987.68  100.000000             Dota 2
167 2025-06-01    411123.31  100.522175             Dota 2
168 2025-07-01    426672.67  104.324089             Dota 2
169 2025-08-01    532355.19  130.164

Heroes of the Storm não tem dados com datas próximas das do League of Legends. Para não perder esses dados, userei ele como exemplo de declinio.